In [137]:
# checking system info
import sys

# for running gradient descent
import numpy as np
import torch
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

# for bringing the data in from R
import pandas as pd


verbose = False

In [138]:
print(sys.executable)

/Users/jseid1/venv311/bin/python


In [139]:

def learnQ(targets, covariates,embedding_dim,verbose):
    # unpacking inputs
    covariate_matrices = covariates
    target_vectors = targets

    # rows (num outcomes)
    Y_1 = covariate_matrices[0]
    m = Y_1.shape[0] 
    num_donors = Y_1.shape[1] 
    # Embedding dimension
    D = embedding_dim

    # Q is what we're optimizing - requires_grad=True tracks gradients
    torch.manual_seed(215)
    Q = torch.randn(num_donors, D, dtype=torch.float64, requires_grad=True)

    # --- Define the inner QP once (structure never changes) ---
    w_var = cp.Variable(D)
    # Create a parameter for each target vector
    YQ_params = [cp.Parameter((m, D)) for _ in range(len(target_vectors))]
    discrepancy = [cp.sum_squares(d.numpy() - YQ_param @ w_var) for YQ_param, d in zip(YQ_params, target_vectors)]
    # I believe this is where I'll add in the many many target and covariate matrices
    constraints = [cp.sum(w_var) == 1, w_var >= 0]
    objective = cp.Minimize(sum(discrepancy))
    prob = cp.Problem(objective, constraints)
    layer = CvxpyLayer(prob, parameters=YQ_params, variables=[w_var])

    # --- Outer optimization loop ---
    optimizer = torch.optim.Adam([Q], lr=0.01)
    l2_norm = sum

    for step in range(2000):
        optimizer.zero_grad()

        # transform the covariates using Q
        YQ_list = [Y @ Q for Y in covariate_matrices]
        # solve for w given the matrix Q
        # * unpacks the list
        w_sol, = layer(*YQ_list) 

        # use l2 norm to regularize Q
        lambda_l2_Q = 0.01 
        lambda_l2_w = 1
        l2_Q = torch.sum(Q**2)
        l2_w = torch.sum(w_sol**2)

        # loss using the optimal w for this Q
        loss = sum(torch.sum((d - YQ @ w_sol)**2) for d, YQ in zip(target_vectors, YQ_list)) + (lambda_l2_Q * l2_Q) + (lambda_l2_w * l2_w)

        # this is where Q is updated
        loss.backward()                 
        optimizer.step()
        
        if step % 200 == 0:
            print(f"Step {step:4d} | Loss: {loss.item():.8f}")

    # --- Results --- #
    if (verbose == True):
        print(f"\nFinal Loss: {loss.item():.8f}")
        print(f"Final w:    {w_sol.detach().numpy().round(4)}")
        print(f"AQ @ w:     {(AQ @ w_sol).detach().numpy().round(4)}")
        print(f"Target d:   {d.numpy()}")
        print(f"Final Q:\n {Q.detach().numpy().round(4)}")
    
    Q_final = Q.detach().numpy()
    w_final = w_sol.detach().numpy()
    return Q_final, w_final
    

## Toy Example

### Toy data set (no underlying patterns)

Notice that the number of time points is greater than the number of units. What if I change this?

In [140]:

# Data
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0], [3.0, 5.0]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0], [4.0, 6.0]], dtype=torch.float64)

d_3 = torch.tensor([4.0, 4.0], dtype=torch.float64)
Y_3 = torch.tensor([[1.0, 3.0], [2.0, 4.0]], dtype=torch.float64)

target_vectors = [d_1, d_2, d_3]
covariate_matrices = [Y_1,Y_2,Y_3]


In [141]:
print(target_vectors[0].shape)   # should be (2,)
print(covariate_matrices[0].shape)  # should be (2, 2)

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 10, False)


print(d_1)
print(Y_1@Q_final@w_final)

print(d_2)
print(Y_2@Q_final@w_final)

print(d_3)
print(Y_3@Q_final@w_final)



torch.Size([2])
torch.Size([2, 2])
Step    0 | Loss: 49.11450626
Step  200 | Loss: 17.00118149
Step  400 | Loss: 15.89905812
Step  600 | Loss: 15.89571876
Step  800 | Loss: 15.64368608
Step 1000 | Loss: 15.60265322
Step 1200 | Loss: 15.60092724
Step 1400 | Loss: 15.59762699
Step 1600 | Loss: 15.59571574
Step 1800 | Loss: 15.59392211
tensor([6., 2.], dtype=torch.float64)
tensor([4.3946, 3.5984], dtype=torch.float64)
tensor([5., 3.], dtype=torch.float64)
tensor([2.1973, 2.8021], dtype=torch.float64)
tensor([4., 4.], dtype=torch.float64)
tensor([5.1909, 4.3946], dtype=torch.float64)


Making the number of donors greater than the number of time points.
This has the effect of greatly decreasing the loss.
And it is also pretty clear that the loss doesn't depend too strongly on the embedding dimension.

In [142]:
# Data
d_1 = torch.tensor([6.0, 2.0], dtype=torch.float64)
Y_1 = torch.tensor([[2.0, 4.0, 1, 9], [3.0, 5.0, 10, 11]], dtype=torch.float64)

d_2 = torch.tensor([5.0, 3.0], dtype=torch.float64)
Y_2 = torch.tensor([[1.0, 2.0, 3, 12], [4.0, 6.0, 5, 7]], dtype=torch.float64)

target_vectors = [d_1, d_2]
covariate_matrices = [Y_1,Y_2]

Q_final, w_final = learnQ(target_vectors, covariate_matrices, 5, False)


print(d_1)
print(Y_1@Q_final@w_final)

print(d_2)
print(Y_2@Q_final@w_final)



Step    0 | Loss: 72.60129912
Step  200 | Loss: 1.08127973
Step  400 | Loss: 0.82281019
Step  600 | Loss: 0.67314261
Step  800 | Loss: 0.57412767
Step 1000 | Loss: 0.52426619
Step 1200 | Loss: 0.50280760
Step 1400 | Loss: 0.49256146
Step 1600 | Loss: 0.48647478
Step 1800 | Loss: 0.48221224
tensor([6., 2.], dtype=torch.float64)
tensor([5.9169, 1.9300], dtype=torch.float64)
tensor([5., 3.], dtype=torch.float64)
tensor([5.0676, 3.0900], dtype=torch.float64)


# Synthetic Data Experiment

- step 1: split into train and test
- step 2 (implement if necessary): normalize
    - Use the mean and STD computed only from the train data (pre-treatment) - not the test data (post treatment!)
    - apply the standardization to both train and test data


In [153]:
# should probably normalize the data in advance, or maybe I don't even need to.

# read in the data
data = pd.read_csv('outcomes.csv')
metadata = pd.read_csv("metadata.csv")

# convert to numpy arrays
metadata = metadata.to_numpy()
data = data.to_numpy()

# unpack the metadata
n_units, n_timepoints, n_outcomes = metadata[:,1]

# reshape the data
Y = data.reshape(n_timepoints, n_outcomes, n_units)

test = Y[len(Y)-1]
train = Y[0:len(Y)-1]

verbose=False
if (verbose):
    print("Test data: \n\n", test)
    print("\nTrain data: \n\n", train)

# print(train[0])
# print(train[0].shape[0],"\n",train[0].shape[1])

# print("\n",train[0][0:2,0:1])
# print("\n",train[0][0:2,1:4])

# split the data into the target vectors and the covariate matrices
train_target_vectors = [torch.from_numpy(train[i][0:train[i].shape[0],0:1]).squeeze() for i in range(len(train))]
train_covariate_matrices = [torch.from_numpy(train[i][0:train[i].shape[0],1:train[i].shape[1]]) for i in range(len(train))]

verbose=False
if verbose == True:
    for i in range(len(train)):
        print("\nTarget vector", i, ":\n", target_vectors[i])
        print("\nCovariate matrix", i, ":\n", covariate_matrices[i])


test_target_vector = torch.from_numpy(test[0:test.shape[0],0:1]).squeeze()

test_covariate_matrix = torch.from_numpy(test[0:test.shape[0],1:test.shape[1]])



## Prior to adding in treatment effect, and even without normalization, the method works well

In [ ]:
# run the learn Q function:
Q_final, w_final = learnQ(train_target_vectors, train_covariate_matrices, 7, False)

print("Computed: \n", test_covariate_matrix@Q_final@w_final)
print("Goal: \n", test_target_vector)


Step    0 | Loss: 1.18645781
Step  200 | Loss: 0.37217840
Step  400 | Loss: 0.36414906
Step  600 | Loss: 0.36238640
Step  800 | Loss: 0.36154081
Step 1000 | Loss: 0.36157857
Step 1200 | Loss: 0.36148355
Step 1400 | Loss: 0.36140522
Step 1600 | Loss: 0.36188253
Step 1800 | Loss: 0.36119444
Computed: 
 tensor([-0.7249, -0.2657], dtype=torch.float64)
Goal: 
 tensor([-0.7178, -0.3063], dtype=torch.float64)
